<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/Time_Series_Forecasting_with_Gaussian_Processes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Install & Import Libraries**

In [ ]:
!pip install gpytorch -qq

import gpytorch
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

### **Data Loading & Preprocessing**

In [ ]:
from google.colab import files
import io
import pandas as pd

# This will create a "Choose Files" button when you run the cell
print("Please upload DailyDelhiClimateTrain.csv:")
uploaded = files.upload()

# Get the actual filename from the uploaded dictionary
# Assuming only one file is uploaded or the desired file is the first one
actual_filename = list(uploaded.keys())[0]

# Read the uploaded file into pandas
df = pd.read_csv(io.BytesIO(uploaded[actual_filename]))

print(df.isnull().sum())

# Convert date to datetime and then to continuous numerical days
df['date'] = pd.to_datetime(df['date'])
df['days_since_start'] = (df['date'] - df['date'].min()).dt.days

X = df['days_since_start'].values.reshape(-1, 1)
y = df['meantemp'].values.reshape(-1, 1)

# Chronological Train-Test Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Normalize Data (Critical for Gaussian Processes)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = torch.tensor(scaler_X.fit_transform(X_train), dtype=torch.float32).squeeze()
y_train_scaled = torch.tensor(scaler_y.fit_transform(y_train), dtype=torch.float32).squeeze()
X_test_scaled = torch.tensor(scaler_X.transform(X_test), dtype=torch.float32).squeeze()
y_test_scaled = torch.tensor(scaler_y.transform(y_test), dtype=torch.float32).squeeze()

### **Model Design**

In [ ]:
# -----------------------------------------------------------
# 2. MODEL DESIGN
# -----------------------------------------------------------
class ClimateGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super(ClimateGPModel, self).__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ConstantMean()

        # Additive Kernel: Periodic (Seasonality) + RBF (Long-term trends)
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.PeriodicKernel()) + \
                            gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood()
model = ClimateGPModel(X_train_scaled, y_train_scaled, likelihood)

# Kernel parameter inspection
print(model.covar_module)
print("Noise:", likelihood.noise.item())

### **Model Training**

In [ ]:
model.train()
likelihood.train()

# Use the Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

# "Loss" for GPs - the marginal log likelihood
mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

print("Starting Training...")
training_iter = 100
for i in range(training_iter):
    optimizer.zero_grad()
    output = model(X_train_scaled)
    loss = -mll(output, y_train_scaled)
    loss.backward()

    if (i+1) % 20 == 0:
        print(f"Iter {i+1}/{training_iter} - Loss: {loss.item():.3f}")

    optimizer.step()

### **Forecasting & Evaluation**

In [ ]:
model.eval()
likelihood.eval()

with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Generate predictions on the test set
    observed_pred = likelihood(model(X_test_scaled))

    # Get predictive mean and confidence (variance)
    pred_mean_scaled = observed_pred.mean
    lower_scaled, upper_scaled = observed_pred.confidence_region()

    # Calculate Log-Likelihood on test set
    log_likelihood = observed_pred.log_prob(y_test_scaled).mean().item()

# Inverse transform to get real-world temperature values
pred_mean = scaler_y.inverse_transform(pred_mean_scaled.numpy().reshape(-1, 1)).flatten()
lower = scaler_y.inverse_transform(lower_scaled.numpy().reshape(-1, 1)).flatten()
upper = scaler_y.inverse_transform(upper_scaled.numpy().reshape(-1, 1)).flatten()

print("\n--- Evaluation Metrics ---")

# Calculate Mean Squared Error
mse = mean_squared_error(y_test.flatten(), pred_mean)

print(f"Test Mean Squared Error (MSE): {mse:.2f} °C\u00b2")
print(f"Test Predictive Log-Likelihood: {log_likelihood:.4f}")

# Calculate RMSE
rmse = np.sqrt(mse)

print(f"RMSE: {rmse:.2f} °C")

# Calculate Mean Absolute Error
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    y_test.flatten(),
    pred_mean
)

print(f"MAE: {mae:.2f}")

### **Visualizing Uncertainty & Forecast**

In [ ]:
plt.figure(figsize=(12, 6))

# Plot historical training data
plt.plot(X_train, y_train, 'k.', markersize=2, label='Training Data', alpha=0.5)

# Plot actual test data
plt.plot(X_test, y_test, 'b.', markersize=4, label='Actual Future Data')

# Plot the forecast (GP Mean)
plt.plot(X_test, pred_mean, 'r-', lw=2, label='GP Forecast Mean')

# Plot the confidence interval (GP Uncertainty)
plt.fill_between(X_test.flatten(), lower, upper, color='red', alpha=0.2, label='95% Confidence Interval')

plt.title('Gaussian Process Time-Series Forecasting: Daily Mean Temperature')
plt.xlabel('Days Since Start')
plt.ylabel('Mean Temperature (°C)')
plt.legend(
    loc='lower center',
    bbox_to_anchor=(0.5, -0.20),
    ncol=4
)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()